# Automated Inter-Rater Reliability (Cohen's Kappa) Workflow
This notebook automates the comparison of behavioral coding CSVs to calculate reliability indices.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score
import matplotlib.pyplot as plt
import seaborn as sns

# Load the sample CSVs provided
file_tayah = '/content/1002_06m_infant_OIX_code-MOIX-ringstacker.csv'
file_felix = '/content/1024_06m_infant_OIX_code-MOIX-noisemaker.csv'

df_tayah = pd.read_csv(file_tayah)
df_felix = pd.read_csv(file_felix)

print("Tayah Data Shape:", df_tayah.shape)
print("Felix Data Shape:", df_felix.shape)

# Display headers to identify the 'behavior' or 'code' columns
display(df_tayah.head())
display(df_felix.head())

Tayah Data Shape: (53, 21)
Felix Data Shape: (133, 21)


,Colors.ordinal,Colors.onset,Colors.offset,Colors.code01,Mouthing.ordinal,Mouthing.onset,Mouthing.offset,Mouthing.code01,Other_Action.ordinal,Other_Action.onset,...,Other_Action.code01,Rhythmic_Action.ordinal,Rhythmic_Action.onset,Rhythmic_Action.offset,Rhythmic_Action.code01,Touching.ordinal,Touching.onset,Touching.offset,Touching.code01,Unnamed: 20
0,0.0,0.0,6922.0,START,0.0,0.0,6922.0,START,0.0,0.0,...,START,0.0,0.0,6922.0,START,0,0,6922,START,NaN
1,1.0,6923.0,9060.0,0,1.0,6923.0,118353.0,0,1.0,29886.0,...,SWIPE,1.0,6923.0,118353.0,0,1,6923,9060,0,NaN
2,2.0,9061.0,10165.0,S,2.0,118354.0,468968.0,END,2.0,29887.0,...,DROP,2.0,118354.0,468968.0,END,2,9061,10165,2,NaN
3,3.0,10166.0,11168.0,0,NaN,NaN,NaN,NaN,3.0,118354.0,...,END,NaN,NaN,NaN,NaN,3,10166,11168,0,NaN
4,4.0,11169.0,11661.0,S,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,4,11169,11661,2,NaN


,Activating.ordinal,Activating.onset,Activating.offset,Activating.code01,Banging.ordinal,Banging.onset,Banging.offset,Banging.code01,Mouthing.ordinal,Mouthing.onset,...,Mouthing.code01,Other_Action.ordinal,Other_Action.onset,Other_Action.offset,Other_Action.code01,Touching.ordinal,Touching.onset,Touching.offset,Touching.code01,Unnamed: 20
0,0.0,0.0,166225.0,START,0.0,0.0,166225.0,START,0.0,0.0,...,START,0.0,0.0,166225.0,START,0,0,166225,START,NaN
1,1.0,175200.0,175200.0,1,1.0,166226.0,168435.0,0,1.0,166226.0,...,0,1.0,172822.0,172822.0,SPIN,1,166226,166463,0,NaN
2,2.0,182342.0,185163.0,1,2.0,168436.0,171257.0,1,2.0,188156.0,...,1,2.0,175882.0,175882.0,SPIN,2,166464,166803,1,NaN
3,3.0,185164.0,185164.0,1,3.0,171258.0,180845.0,0,3.0,190808.0,...,0,3.0,183872.0,183872.0,SPIN,3,166804,167041,0,NaN
4,4.0,201688.0,201688.0,2,4.0,180846.0,182477.0,1,4.0,225352.0,...,1,4.0,184620.0,184620.0,SPIN,4,167042,167143,1,NaN


### Reusable Automation Workflow
The function below automates the entire pipeline for any new pair of CSV files.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score, accuracy_score
import os

def calculate_coder_reliability(file_path_a, file_path_b, output_prefix="Reliability"):
    """
    Improved reliability script: pairwise comparison, time-point aware,
    detailed disagreement logging, and simplified user input.
    """
    def load_and_clean(path):
        df = pd.read_csv(path)
        # Robust column detection
        potential_cols = [c for c in df.columns if any(k in c.upper() for k in ['CODE', 'TOY', 'PARADIGM'])]
        if not potential_cols:
            raise ValueError(f"Could not find a coding column in {path}.")
        target_col = potential_cols[0]

        is_9m = '9m' in path.lower()
        d = df[['time', target_col]].copy()
        d.columns = ['time', 'code']
        return d, is_9m

    # 1. Load Data
    try:
        d1, is_9m_a = load_and_clean(file_path_a)
        d2, is_9m_b = load_and_clean(file_path_b)
    except Exception as e:
        return f"Error: {e}"

    # 2. Alignment
    merged = pd.merge_asof(d1.sort_values('time'), d2.sort_values('time'), on='time', direction='nearest', suffixes=('_A', '_B'))
    merged = merged.dropna(subset=['code_A', 'code_B'])

    # 3. Standardization Mapping (OIX Master List + Numeric support)
    def normalize(val, is_9m):
        if pd.isna(val): return "MISSING"

        # Convert numeric codes to strings if necessary (e.g., 1.0 -> '1')
        val_str = str(val).strip().upper().split('.')[0]

        # Mapping both text and potential numeric IDs
        norm_map = {
            '1': 'RING', 'RING': 'RING',
            '2': 'SQUIGGLE', 'SQUIGGLE': 'SQUIGGLE', 'SQUIGGLE2': 'SQUIGGLE',
            '3': 'NOISEMAKER', 'NOISEMAKER': 'NOISEMAKER',
            '4': 'BLOCK', 'BLOCK': 'BLOCK',
            '5': 'CAR', 'CAR': 'CAR',
            '6': 'DRUM', 'DRUM': 'DRUM',
            '0': 'NO_TOY', 'NO_TOY': 'NO_TOY', 'NONE': 'NO_TOY'
        }

        # Time-specific Mappings (FISH vs POP)
        if is_9m:
            if 'FISH' in val_str or val_str == '7': return 'FISH'
        else:
            if 'POP' in val_str or val_str == '7': return 'POP'

        return norm_map.get(val_str, val_str)

    merged['Clean_A'] = merged['code_A'].apply(lambda x: normalize(x, is_9m_a))
    merged['Clean_B'] = merged['code_B'].apply(lambda x: normalize(x, is_9m_b))

    # 4. Metrics
    y1, y2 = merged['Clean_A'], merged['Clean_B']
    kappa = cohen_kappa_score(y1, y2)
    agreement = accuracy_score(y1, y2)
    disagreement = 1 - agreement

    # 5. Identify Disagreement Locations
    disagreements = merged[merged['Clean_A'] != merged['Clean_B']].copy()
    disagreements = disagreements[['time', 'code_A', 'code_B', 'Clean_A', 'Clean_B']]

    # 6. Output Results
    summary = pd.DataFrame({
        'File_A': [os.path.basename(file_path_a)],
        'File_B': [os.path.basename(file_path_b)],
        'Cohen_Kappa': [round(kappa, 4)],
        'Percent_Agreement': [f"{agreement:.2%}"],
        'Percent_Disagreement': [f"{disagreement:.2%}"],
        'Disagreement_Count': [len(disagreements)],
        'Total_Samples': [len(merged)]
    })

    summary.to_csv(f"{output_prefix}_Summary.csv", index=False)
    disagreements.to_csv(f"{output_prefix}_Disagreement_Log.csv", index=False)

    print(f"--- Reliability Report ---")
    display(summary)
    if not disagreements.empty:
        print(f"\n--- Disagreement Details (Sample) ---")
        display(disagreements.head())

    return summary, disagreements

# --- USER SECTION ---
FILE_1 = '/content/1038_12m_OIX_Felix.csv'
FILE_2 = '/content/1038_12m_OIX_Tayah.csv'

summary_report, diff_log = calculate_coder_reliability(FILE_1, FILE_2)

--- Reliability Report ---


,File_A,File_B,Cohen_Kappa,Percent_Agreement,Percent_Disagreement,Disagreement_Count,Total_Samples
0,1038_12m_OIX_Felix.csv,1038_12m_OIX_Tayah.csv,0.8425,87.37%,12.63%,1590,12594



--- Disagreement Details (Sample) ---


,time,code_A,code_B,Clean_A,Clean_B
3375,3691217,2.0,1.0,SQUIGGLE,RING
9395,3932017,5.0,4.0,CAR,BLOCK
11006,3996457,6.0,5.0,DRUM,CAR
11007,3996497,6.0,5.0,DRUM,CAR
11008,3996537,6.0,5.0,DRUM,CAR
